# P4a — Rule-Based Regex Baseline (M0)

## 1. Introduction and Objective

I start with a simple rule-based classifier as a baseline for predicting ICD-defined low back pain from ED chief complaints. Miotto et al. (2020) used keyword search as one of their methods for identifying acute LBP from clinical notes. Weidman et al. (2026) further demonstrated that even very short chief complaints can contain meaningful lexical information for clinical prediction.

Drawing on these studies, I define a limited set of keyword patterns based on the literature, NCQA medical terminology, and exploratory analysis of the training data. The objective is to establish a reasonable and interpretable lexical baseline against which more sophisticated NLP models can be compared, rather than to develop a comprehensive clinical rule system.

## 2. Development Dataset Preparation

I use the modeling dataset and fixed patient-grouped partitions established in Notebook 03. The cleaned chief complaint is the input, and the ICD-derived binary LBP label is the prediction target.

The training set is used for keyword exploration and rule development, while the validation set is reserved for evaluating the resulting classifier. The test set remains held out until the final comparative evaluation.

In [2]:
from pathlib import Path
import pandas as pd

# Load the dataset and fixed partition assignments from P3
DATA_DIR = Path("../results/03_experimental_setup")

df = pd.read_csv(
    DATA_DIR / "modeling_dataset.csv",
    keep_default_na=False
)
splits = pd.read_csv(DATA_DIR / "dataset_splits.csv")

# Retain development partitions only
dev_splits = splits[splits["split"].isin(["train", "validation"])]

dev_df = df.merge(
    dev_splits,
    on="stay_id",
    how="inner",
    validate="one_to_one"
)

train_df = dev_df[dev_df["split"] == "train"].copy()
val_df = dev_df[dev_df["split"] == "validation"].copy()

# Check development dataset integrity
assert len(dev_df) == len(dev_splits)
assert dev_df["chiefcomplaint_clean"].notna().all()
assert dev_df["lbp_label"].isin([0, 1]).all()

# Summarize development partitions
summary = dev_df.groupby("split").agg(
    encounters=("stay_id", "size"),
    patients=("subject_id", "nunique"),
    positives=("lbp_label", "sum")
)

summary["negatives"] = summary["encounters"] - summary["positives"]
summary["prevalence (%)"] = (
    summary["positives"] / summary["encounters"] * 100
).round(2)

display(summary)

,encounters,patients,positives,negatives,prevalence (%)
split,,,,,
train,295877,143119,11429,284448,3.86
validation,42268,20483,1632,40636,3.86


## 3. Keyword Selection and Lexical Analysis

Rather than choosing keywords arbitrarily, I draw on three sources: existing literature, standardized medical terminology, and lexical patterns observed in the training data.

Miotto et al. (2020) provide a starting point for keyword-based LBP identification. The NCQA ICD and SNOMED CT descriptions serve as additional vocabulary references. Inspired by Weidman et al. (2026), I also examine the occurrence of words and phrases across the positive and negative classes.

These sources guide the manual selection of a limited set of keyword patterns for the regex baseline.

**Initial Keyword Candidates**

| Candidate | Source | Rationale |
|---|---|---|
| low back pain | Miotto; NCQA | Direct expression of LBP |
| lower back pain | Adapted from low back pain | Common wording variation |
| lbp | Miotto | Abbreviation for low back pain |
| back pain | Miotto | Broader symptom description |
| low bp | Miotto | Potentially ambiguous abbreviation |
| lumbago | NCQA ICD-9-CM / ICD-10-CM | Standard medical term |
| sciatica | NCQA ICD / SNOMED CT | Related clinical expression |
| backache | NCQA ICD-9-CM | Alternative expression of back pain |
| low back strain | NCQA SNOMED CT | Related musculoskeletal condition |
| lumbar sprain | NCQA SNOMED CT | Related musculoskeletal condition |

In [3]:
import re

# Candidate keywords from literature and NCQA terminology
candidates = [
    "low back pain",
    "lower back pain",
    "lbp",
    "back pain",
    "low bp",
    "lumbago",
    "sciatica",
    "backache",
    "low back strain",
    "lumbar sprain"
]

# Separate training encounters by reference label
positive_text = train_df.loc[
    train_df["lbp_label"] == 1, "chiefcomplaint_clean"
]

negative_text = train_df.loc[
    train_df["lbp_label"] == 0, "chiefcomplaint_clean"
]

# Compare keyword occurrence across the two classes
records = []

for term in candidates:
    pattern = rf"\b{re.escape(term)}\b"

    pos_count = positive_text.str.contains(
        pattern, case=False, regex=True
    ).sum()

    neg_count = negative_text.str.contains(
        pattern, case=False, regex=True
    ).sum()

    pos_rate = pos_count / len(positive_text)
    neg_rate = neg_count / len(negative_text)

    records.append({
        "keyword": term,
        "positive_count": pos_count,
        "negative_count": neg_count,
        "positive_coverage (%)": pos_rate * 100,
        "negative_coverage (%)": neg_rate * 100,
        "coverage_difference (pp)": (pos_rate - neg_rate) * 100
    })

lexical_stats = (
    pd.DataFrame(records)
    .sort_values("coverage_difference (pp)", ascending=False)
    .reset_index(drop=True)
)

display(lexical_stats.round(2))

,keyword,positive_count,negative_count,positive_coverage (%),negative_coverage (%),coverage_difference (pp)
0,back pain,7766,3919,67.95,1.38,66.57
1,lower back pain,2970,849,25.99,0.30,25.69
2,lbp,195,38,1.71,0.01,1.69
3,low back pain,125,32,1.09,0.01,1.08
4,sciatica,11,1,0.10,0.00,0.10
5,backache,1,0,0.01,0.00,0.01
6,lumbago,0,0,0.00,0.00,0.00
7,low back strain,0,0,0.00,0.00,0.00
8,lumbar sprain,0,0,0.00,0.00,0.00
9,low bp,0,8,0.00,0.00,-0.00


**Observation and Keyword Selection**

The phrase "back pain" provides the strongest lexical signal, appearing in 67.95% of positive encounters compared with 1.38% of negative encounters. More specialized medical terms are much less common, suggesting that standardized medical terminology does not necessarily reflect the language used in ED chief complaints.

I retain five keywords: back pain, lbp, sciatica, backache, and lumbago. The phrases "low back pain" and "lower back pain" are already covered by "back pain". Although lumbago is not observed in the training data, I retain it as an established medical term for LBP. The ambiguous abbreviation "low bp" and the unobserved injury-related phrases are excluded.

## 4. Regex Implementation and Rule Refinement

Based on the keyword analysis in Section 3, I construct a case-insensitive regex classifier using five selected keywords: back pain, lbp, sciatica, backache, and lumbago. Word boundaries are used to avoid unintended substring matches. A chief complaint is classified as positive if it contains any of these keywords and negative otherwise.

I first verify the matching behavior using synthetic examples and then evaluate the classifier on the training set to determine whether any basic rule adjustments are needed.

In [4]:
# Define the selected keywords
keywords = [
    "back pain",
    "lbp",
    "sciatica",
    "backache",
    "lumbago"
]

# Construct and compile the regex pattern
pattern = re.compile(
    r"\b(?:" + "|".join(re.escape(k) for k in keywords) + r")\b",
    flags=re.IGNORECASE
)

def predict_regex(text):
    return int(bool(pattern.search(text)))

# Verify matching behavior using synthetic examples
test_cases = [
    ("back pain", 1),
    ("Lower Back Pain", 1),
    ("acute low back pain", 1),
    ("LBP", 1),
    ("sciatica", 1),
    ("backache", 1),
    ("lumbago", 1),
    ("upper back pain", 1),
    ("abdominal pain", 0),
    ("low BP", 0),
    ("headache", 0),
    ("lbpx", 0)
]

unit_tests = pd.DataFrame(
    test_cases, columns=["chief_complaint", "expected"]
)

unit_tests["predicted"] = unit_tests["chief_complaint"].apply(predict_regex)
unit_tests["passed"] = unit_tests["expected"] == unit_tests["predicted"]

display(unit_tests)

assert unit_tests["passed"].all()
print("Regex unit tests: PASSED")

,chief_complaint,expected,predicted,passed
0,back pain,1,1,True
1,Lower Back Pain,1,1,True
2,acute low back pain,1,1,True
3,LBP,1,1,True
4,sciatica,1,1,True
5,backache,1,1,True
6,lumbago,1,1,True
7,upper back pain,1,1,True
8,abdominal pain,0,0,True
9,low BP,0,0,True


Regex unit tests: PASSED


In [5]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Apply the regex classifier to the training set
y_train = train_df["lbp_label"]
y_train_pred = train_df["chiefcomplaint_clean"].apply(predict_regex)

# Evaluate training performance
train_metrics = pd.DataFrame([{
    "precision": precision_score(y_train, y_train_pred, zero_division=0),
    "recall": recall_score(y_train, y_train_pred, zero_division=0),
    "f1": f1_score(y_train, y_train_pred, zero_division=0)
}], index=["M0 - Regex"])

display(train_metrics.round(4))

# Confusion matrix
train_cm = pd.DataFrame(
    confusion_matrix(y_train, y_train_pred, labels=[0, 1]),
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)

display(train_cm)

,precision,recall,f1
M0 - Regex,0.6683,0.6976,0.6826


,Predicted Negative,Predicted Positive
Actual Negative,280490,3958
Actual Positive,3456,7973


**Observation**

The regex classifier achieves a precision of 0.6683, a recall of 0.6976, and an F1 score of 0.6826 on the training set.

Interestingly, most of its predictive ability comes from a single phrase, "back pain", which alone identifies 7,766 positive encounters. The other four keywords collectively identify only 207 additional positive encounters and 39 additional negative encounters. This suggests that much of the predictive information in these short chief complaints may be captured by a few simple lexical patterns, consistent with the motivation behind Weidman et al. (2026).

Given the intended simplicity of this baseline, I retain the current rules without further refinement. It will be interesting to see whether more sophisticated NLP models can capture additional predictive information beyond these obvious lexical signals.

## 5. Validation Performance

I evaluate the finalized regex classifier on the fixed validation set using precision, recall, F1 score, and the confusion matrix. The same five keyword patterns are applied without further modification.

The validation results provide an initial assessment of how well the rules generalize beyond the training data. The test set remains held out for the final model comparison.

In [6]:
# Apply the finalized regex classifier to the validation set
y_val = val_df["lbp_label"]
y_val_pred = val_df["chiefcomplaint_clean"].apply(predict_regex)

# Evaluate validation performance
val_metrics = pd.DataFrame([{
    "precision": precision_score(y_val, y_val_pred, zero_division=0),
    "recall": recall_score(y_val, y_val_pred, zero_division=0),
    "f1": f1_score(y_val, y_val_pred, zero_division=0)
}], index=["M0 - Regex"])

# Compare training and validation performance
performance_summary = pd.concat([
    train_metrics.rename(index={"M0 - Regex": "Train"}),
    val_metrics.rename(index={"M0 - Regex": "Validation"})
])

display(performance_summary.round(4))

# Validation confusion matrix
val_cm = pd.DataFrame(
    confusion_matrix(y_val, y_val_pred, labels=[0, 1]),
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"]
)

display(val_cm)

,precision,recall,f1
Train,0.6683,0.6976,0.6826
Validation,0.6686,0.6924,0.6803


,Predicted Negative,Predicted Positive
Actual Negative,40076,560
Actual Positive,502,1130


The regex classifier achieves a validation F1 score of 0.6803, compared with 0.6826 on the training set. Precision and recall also remain similar across the two partitions, suggesting that the selected keyword patterns generalize well to unseen encounters. This establishes a simple yet competitive lexical baseline for the subsequent NLP model comparison. No further rule adjustments are made.

I save the finalized regex pattern and validation results for the subsequent model comparison. The rules remain unchanged for the final test evaluation.

In [7]:
import json

OUTPUT_DIR = Path("../results/04a_regex_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the finalized regex configuration
regex_config = {
    "model_id": "M0",
    "model_name": "Regex Baseline",
    "keywords": keywords,
    "pattern": pattern.pattern,
    "ignore_case": True,
    "input_column": "chiefcomplaint_clean"
}

with open(OUTPUT_DIR / "m0_regex_config.json", "w") as f:
    json.dump(regex_config, f, indent=4)

# Save validation metrics and confusion matrix counts
tn, fp, fn, tp = val_cm.to_numpy().ravel()

results = val_metrics.copy()
results["tn"] = tn
results["fp"] = fp
results["fn"] = fn
results["tp"] = tp

results.to_csv(
    OUTPUT_DIR / "m0_validation_metrics.csv",
    index_label="model"
)

print("M0 configuration and validation results saved.")

M0 configuration and validation results saved.
